In [ ]:
# Last Update: 6/14/2026 ; 9:33pm

In [48]:
# Configuration cell

import os
import numpy as np
from scipy.io import loadmat
import matplotlib.pyplot as plt
import pywt
import scipy.stats as stats
import pandas as pd
from scipy.interpolate import UnivariateSpline
from scipy.signal import butter, filtfilt, resample
from scipy.interpolate import griddata
import json
from collections import Counter
import mne


# Data Path 
BASE_PATH = r"D:\Research\EEG-fNIRS\project\project\Dataset"

# Subject setting
SUBJECT_START = 1  # First Subject
SUBJECT_END = 2    # Last Subject

# Processing Settings
SAVE_OUTPUT = True 
PLOT_FIGURES = True
VERBOSE = True 
APPLY_ICA = True

# Data Settings
FS_EEG = 200.0 
FS_NIRS = 10.0
EPOCH_TMIN = 0.0   # Epoch onset (sec)
EPOCH_TMAX = 3.0   # Epoch End (sec)

# Subject List
def get_subject_list():
    return [f"VP{num:03d}" for num in range(SUBJECT_START, SUBJECT_END + 1)]

SUBJECTS_TO_PROCESS = get_subject_list()

In [6]:
# Checking real markers in the dataset

def inspect_markers(subject_id, base_path=BASE_PATH):

    print(f"Subject: {subject_id}")
    print(f"Path: {base_path}")

    eeg_path = os.path.join(base_path, f"{subject_id}-EEG", "mrk_nback.mat")

    if not os.path.exists(eeg_path):
        print(f"no EEG file found")
        print(f"Path: {eeg_path}")
        return None, None
        
    # EEG markers
    mrk_eeg = loadmat(eeg_path)['mrk_nback'][0, 0]
    eeg_codes = mrk_eeg['event'][0, 0]['desc'].flatten().astype(int)
    eeg_times = mrk_eeg['time'].flatten()

    print("=" * 60)
    print(f"Total number of EEG markers: {len(eeg_codes)}")
    print(f"Number of unique EEG markers: {sorted(np.unique(eeg_codes))}")
    
    unique, counts = np.unique(eeg_codes, return_counts=True)
    for code, count in zip(unique, counts):
        desc = {
            16: "0-back target",
            48: "2-back target", 
            64: "2-back non-target",
            80: "3-back target",
            96: "3-back non-target",
            112: "0-back session start",
            128: "2-back session start",
            144: "3-back session start"
        }.get(code, "unknown")
        
        print(f"   marker {code}: {count} times ({desc})")


    # fNIRS markers
    fnirs_path = os.path.join(base_path, f"{subject_id}-NIRS", "mrk_nback.mat")
    print(f"\n fNIRS path: {fnirs_path}")

    if not os.path.exists(fnirs_path):
        print(f"no fNIRS file found")
        print(f"Path: {fnirs_path}")
        return None, None
        
    mrk_fnirs = loadmat(fnirs_path)['mrk_nback'][0, 0]
    fnirs_codes = mrk_fnirs['event'][0, 0]['desc'].flatten().astype(int)
    fnirs_times = mrk_fnirs['time'].flatten()

    print("=" * 60)
    print(f"Total number of fNIRS markers: {len(fnirs_codes)}")
    print(f"Number of unique fNIRS markers: {sorted(np.unique(fnirs_codes))}")

    for code, count in zip(*np.unique(fnirs_codes, return_counts=True)):
        desc = {
            7: "0-back session start",
            8: "2-back session start", 
            9: "3-back session start"
        }.get(code, "unknown")
        
        print(f"   marker {code}: {count} times ({desc})")

    print("Cheching markers is successfully COMPELTED!")
    
    return eeg_codes, fnirs_codes

first_subject = SUBJECTS_TO_PROCESS[0]
eeg_markers, fnirs_markers = inspect_markers(first_subject, base_path=BASE_PATH)


Subject: VP001
Path: D:\Research\EEG-fNIRS\project\project\Dataset
no EEG file found
Path: D:\Research\EEG-fNIRS\project\project\Dataset\VP001-EEG\mrk_nback.mat


In [9]:
# Loading the dataset and markers (BASED ON THE DATA DESCRIPTION)

def load_data_with_markers(subject_id: str, base_path=BASE_PATH):
    
    subject_eeg = f"{subject_id}-EEG"

    print(f"Loading Data {subject_id}")
    print(f"Path: {base_path}")

    # Loading files
    eeg_folder = os.path.join(base_path, subject_eeg)
    cnt_path = os.path.join(eeg_folder, "cnt_nback.mat")
    mrk_path = os.path.join(eeg_folder, "mrk_nback.mat")
    mnt_path = os.path.join(eeg_folder, "mnt_nback.mat")
    
    for path in [cnt_path, mrk_path, mnt_path]:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Folder is not found: {path}")

    # Raw EEG Data
    cnt_data = loadmat(cnt_path)
    cnt_struct = cnt_data['cnt_nback'][0, 0]
    sfreq = float(cnt_struct['fs'][0, 0]) #Sampling rate from data description
    ch_names = [name[0] for name in cnt_struct['clab'][0]]

    raw_data = cnt_struct['x'].T / 1e6    #microvolt to volt 

    print(f"EEG Sampling rate: {sfreq} Hz")
    print(f"Number of channels: {len(ch_names)}")
    print(f"Number of samples {raw_data.shape[1]:,}")

    # Marker extraction from mrk_nback
    mrk_data = loadmat(mrk_path)
    mrk_struct = mrk_data['mrk_nback'][0, 0]
    event_times_ms = mrk_struct['time'].flatten()  # time (msec)
    event_codes = mrk_struct['event'][0, 0]['desc'].flatten().astype(int)
    
    event_samples = (event_times_ms / 1000 * sfreq).astype(int) # time to sample

    events = np.vstack([event_samples, np.zeros_like(event_samples), event_codes]).T  # Event MNE format: [sample, 0, event_code]

    print(f"Number of Events: {len(events)}")
    print(f"Existing Events: {sorted(np.unique(event_codes))}")

    # Raw MNE
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(raw_data, info, verbose=False)

    # Adding the reference channel (TP9 based on description)
    mne.add_reference_channels(raw, ref_channels='TP9', copy=False)

    # Channel positioning for brain mapping
    mnt_data = loadmat(mnt_path)
    mnt_struct = mnt_data['mnt_nback'][0, 0]
    pos_3d_raw = mnt_struct['pos_3d']  # Channels 3D positions

    ch_pos_dict = {}
    for name, pos in zip(ch_names, pos_3d_raw.T):
        ch_pos_dict[name] = pos

    raw.set_montage(mne.channels.make_dig_montage(ch_pos=ch_pos_dict), 
                    on_missing='ignore')

    # Setting EOG channel types
    raw.set_channel_types({'HEOG': 'eog', 'VEOG': 'eog'}, verbose=False)


    print("Loading is now COMPLETED!")

    return raw, events, sfreq

## Testing for subject 001
#if __name__ == "__main__":
#    raw, events, sfreq = load_data_with_markers(subject_id)
#    
#    print("\n" + "=" * 60)
#    print(f"   Subject: {subject_id}")
#    print(f"   Sampling rate: {sfreq} Hz")
#    print(f"   number of EEg channels: {len(raw.ch_names)}")
#    print(f"   Number of events: {len(events)}")
#    print(f"   Data timing range: {raw.times[0]:.1f} - {raw.times[-1]:.1f} sec")
#    print("=" * 60)

In [10]:
# Marker Definitions

def get_marker_definitions():

    # Trial Markers
    trial_markers = {
        16: 0,   # 0-back target     (Cognitive Workload 0)
        48: 1,   # 2-back target     (Cognitive Workload 1)
        64: 1,   # 2-back non-target (Cognitive Workload 1)
        80: 2,   # 3-back target    (Cognitive Workload 2)
        96: 2,   # 3-back non-target (Cognitive Workload 2)
    }

    # Session Markers
    session_markers = {
        112: 0,   # 0-back session
        128: 1,   # 2-back session
        144: 2,   # 3-back session
    }

    # MNE epochs
    mne_event_id = {f"trial_{code}": code for code in trial_markers.keys()}
    
    return trial_markers, session_markers, mne_event_id

# Printing Definitions
trial_markers, session_markers, mne_event_id = get_marker_definitions()
print("Trial Markers:", trial_markers)
print("Session Markers:", session_markers)



Trial Markers: {16: 0, 48: 1, 64: 1, 80: 2, 96: 2}
Session Markers: {112: 0, 128: 1, 144: 2}


In [12]:
# Epoch extraction
# Each trial: 0.5s number display + 1.5s fixation cross = 2s total 
# so for cognitive load analysis: 3-second window after the trial onset

def extract_epochs_corrected(raw, events, sfreq, tmin=0.0, tmax=3.0):

    trial_markers, session_markers, mne_event_id = get_marker_definitions()

    print("Epoch Extraction")
    print(f"Timing range: {tmin} - {tmax} sec after trial onset")
    print(f"epoch length: {tmax - tmin} sec")
    print(f"Number of samples per epoch: {int((tmax - tmin) * sfreq)}")

    # extrcat epochs for all trial markers 
    trial_event_ids = list(trial_markers.keys())

    print(f"trial events {trial_event_ids}")

    epochs = mne.Epochs(
        raw, 
        events, 
        event_id=trial_event_ids,  # for trial events
        tmin=tmin, 
        tmax=tmax,
        baseline=None,
        preload=True, 
        verbose=False
    )

    print(f"Number of extraction epochs: {len(epochs)}")
    print(f"Data shape: {epochs.get_data().shape}")

    # Assigning cognitive load labels to each epoch based on session
    # Finding session's onset:
    session_starts = []
    for event in events:
        if event[2] in session_markers:
            session_starts.append({
                'sample': event[0],
                'time': event[0] / sfreq,
                'label': session_markers[event[2]]
            })
    session_starts.sort(key=lambda x: x['sample'])

    print(f"Number of sessions found: {len(session_starts)}")
    print(f"session onset time (sec): {[s['time'] for s in session_starts[:5]]}...")
    
    # Label assignment
    labels = []
    trial_events = epochs.events
    unassigned_count = 0

    for event in trial_events:
        trial_time = event[0] / sfreq
        trial_code = event[2]
    
        # Initial label from trial_markers (for verification)
        initial_label = trial_markers.get(trial_code, -1)
        
        assigned_label = -1

        # Finding the session
        for i, sess in enumerate(session_starts):
            if i + 1 < len(session_starts):
                next_time = session_starts[i+1]['time']
            else:
                next_time = float('inf')
            
            if sess['time'] <= trial_time < next_time:
                assigned_label = sess['label']
                break
                
        # Checking consistency between trial and session tags
        if assigned_label != initial_label and initial_label != -1:
            print(f"trial with code {trial_code} is in session with label {assigned_label}" 
                  f"But the trial label is {initial_label}")
        
        if assigned_label == -1:
            unassigned_count += 1
            
        labels.append(assigned_label)
        
    labels = np.array(labels)

    print(f"\n number of trials without label: {unassigned_count} from {len(labels)}")

    # Final Stats
    valid_labels = labels[labels != -1]
    if len(valid_labels) > 0:
        unique_labels, counts = np.unique(valid_labels, return_counts=True)
        for lbl, cnt in zip(unique_labels, counts):
            workload_name = {0: "0-back", 1: "2-back", 2: "3-back"}.get(lbl, "Unknown")
            print(f"      {workload_name}: {cnt} trial ({cnt/len(valid_labels)*100:.1f}%)")
    else:
        print(f"time of the first trial: {trial_events[0][0] / sfreq:.1f}s")
        print(f"time of the first session: {session_starts[0]['time']:.1f}s")

    return epochs, labels


## Testing this cell for a subject
#if __name__ == "__main__":

#    raw, events, sfreq = load_data_with_markers("VP001", base_path=BASE_PATH)
#    epochs, labels = extract_epochs_corrected(raw, events, sfreq, tmin=0.0, tmax=3.0)

#    for i in range(min(5, len(labels))):
#        if labels[i] != -1:
#            workload_name = {0: "0-back", 1: "2-back", 2: "3-back"}.get(labels[i], "Unknown")
#            print(f"   Trial {i+1}: label = {labels[i]} ({workload_name})")
#        else:
#            print(f"   Trial {i+1}: label = {labels[i]} (unlabeled)")
            
#    valid_labels = labels[labels != -1]
#    if len(valid_labels) > 0:
#        print(f"   unlabled distributions: {np.bincount(valid_labels)}")
#    else:
#        print(f"   all {len(labels)} epochs are unlabeled!")
#    print(f"  data strcuture: {epochs.get_data().shape}")


In [ ]:
!pip install mne-icalabel

In [14]:
# Artifact removal using ICA

import mne_icalabel

def preprocess_eeg_pipeline(raw, apply_ica=True, verbose=False):
    
    raw_clean = raw.copy()

    # 1 Hz high-pass filter to remove drift
    raw_clean.filter(l_freq=1.0, h_freq=None, method='fir', 
                     phase='zero', fir_window='hamming', verbose=verbose)

    # Power line noise removal
    raw_clean.notch_filter(freqs=50.0, verbose=verbose)

    # ICA with Picard method
    ica = None
    if apply_ica:
        # Selecting only EEG channels for ICA (no EOG)
        picks_eeg = mne.pick_types(raw_clean.info, eeg=True, exclude='bads')
        raw_for_ica = raw_clean.copy().pick(picks_eeg)

        if 'TP9' in raw_for_ica.ch_names:
            raw_for_ica.drop_channels(['TP9'])
            print("Channel TP9 has been removed for ICA")
            
        raw_for_ica_filtered = raw_for_ica.copy()
        raw_for_ica_filtered.filter(l_freq=1.0, h_freq=99.0, method='fir',
                                     phase='zero', verbose=verbose)

        ica = mne.preprocessing.ICA(n_components=0.99, method='infomax',
                                    fit_params=dict(extended=True),
                                    random_state=42, verbose=verbose)
        ica.fit(raw_for_ica_filtered)

        eog_indices, eog_scores = ica.find_bads_eog(
            raw_clean, ch_name='VEOG', threshold=2.0, verbose=verbose
        )

        ic_labels = mne_icalabel.label_components(raw_for_ica_filtered, ica, method='iclabel')

        non_brain_labels = ['eye blink', 'eye movement', 'muscle artifact', 'line noise']
        artifact_idx = [idx for idx, label in enumerate(ic_labels['labels']) 
                        if label in non_brain_labels]

        if verbose:
            print(f"Identified partions for removal:")
            all_excludes = sorted(set(eog_indices) | set(artifact_idx))
            if all_excludes: 
                for idx in all_excludes:
                    label = ic_labels['labels'][idx]
                    print(f"         - Component ICA{idx:03d}: {label}")
                else:
                    print("no artifact is found")

        ica.exclude = list(set(eog_indices) | set(artifact_idx))

        print(f"number of removed sections: {len(ica.exclude)}")

        if len(ica.exclude) > 0:
            ica.apply(raw_clean, exclude=ica.exclude, verbose=verbose)
        else:
            print("no artifact is found")
            
    # 45Hz low pass filter
    raw_clean.filter(l_freq=None, h_freq=45.0, method='fir',
                     phase='zero', fir_window='hamming', verbose=verbose)

    # referencing to average
    raw_clean.set_eeg_reference('average', projection=False, verbose=verbose)

    print("Prepprocessing is now COMPLETED!")

    return raw_clean, ica

## Testing preprocessing 
#if __name__ == "__main__":

#    raw, events, sfreq = load_data_with_markers("VP001", base_path=BASE_PATH)
#    raw_clean, ica = preprocess_eeg_pipeline(raw, apply_ica=True, verbose=True)
#    epochs_clean, labels_clean = extract_epochs_corrected(raw_clean, events, sfreq, tmin=0.0, tmax=3.0)

#    print(f"number fo epochs: {len(epochs_clean)}")
#    print(f"data strcuture: {epochs_clean.get_data().shape}")
#    print(f"labels distibution: {np.bincount(labels_clean)}")
    

In [20]:
def save_preprocessed_data(epochs, labels, subject_id, output_dir="PREPROCESSED"):

    os.makedirs(output_dir, exist_ok=True)

    print("Saving preprocessed sample {subject_id}")
    
    # Extarcting data from epochs
    data = epochs.get_data()  # (n_trials, n_channels, n_times)
    ch_names = epochs.ch_names

    # saving as numpy
    np.save(f'{output_dir}/{subject_id}_epochs_data.npy', data)
    print(f"data is saved: {output_dir}/{subject_id}_epochs_data.npy")
    print(f"data structure: {data.shape}")
    
    np.save(f'{output_dir}/{subject_id}_labels.npy', labels)
    print(f"labels are saved: {output_dir}/{subject_id}_labels.npy")
    print(f"distibution: {np.bincount(labels)}")
    
    # channel infromation saving
    with open(f'{output_dir}/{subject_id}_ch_names.txt', 'w') as f:
        for ch in ch_names:
            f.write(f"{ch}\n")

    # saving MNE epochs
    epochs.save(f'{output_dir}/{subject_id}_epochs-epo.fif', overwrite=True)

    print(f"data from {subject_id} is now SAVED!")
    print(f"data strcuture: {data.shape}")
    print(f"label distibution: {np.bincount(labels)}")
    print(f"directory: {output_dir}/")

# Saving preprocessed data
#save_preprocessed_data(epochs_clean, labels_clean, "VP001", output_dir="PREPROCESSED")
    

In [21]:
# A comprehensive function for SUBJECT procesing

def process_one_subject(subject_id: str, save_output=True, plot_figures=True):

    print(f"Processing of {subject_id}")

    # Loading data
    raw, events, sfreq = load_data_with_markers(subject_id, base_path=BASE_PATH)

    # Preprocessing
    raw_clean, ica = preprocess_eeg_pipeline(raw, apply_ica=True, verbose=False)

    # Extracting epochs
    epochs, labels = extract_epochs_corrected(raw_clean, events, sfreq, tmin=0.0, tmax=3.0)

    # Saving results
    if save_output:
        save_preprocessed_data(epochs, labels, subject_id)

    # Plotting QC
    if plot_figures:

        print("\n QC plots")

        # PSD from pre- and post-analysis
        fig_psd_before = raw.compute_psd(fmax=50).plot(average=True, show=False)
        fig_psd_before.axes[0].set_title(f'PSD Before Preprocessing - {subject_id}')
        fig_psd_before.savefig(f'{subject_id}_PSD_before.png', dpi=150)
        plt.close(fig_psd_before)
        
        fig_psd_after = raw_clean.compute_psd(fmax=50).plot(average=True, show=False)
        fig_psd_after.axes[0].set_title(f'PSD After Preprocessing - {subject_id}')
        fig_psd_after.savefig(f'{subject_id}_PSD_after.png', dpi=150)
        plt.close(fig_psd_after)

        # ERP average
        fig_erp = epochs.average().plot(spatial_colors=True, show=False)
        fig_erp.savefig(f'{subject_id}_ERP_average.png', dpi=150)
        plt.close(fig_erp)
        
    print(f"Processing of Sample {subject_id} is successfully COMPELTED!")

    return epochs, labels, raw_clean


# Testing the whole process for one subject
#if __name__ == "__main__":
#    epochs, labels, raw_clean = process_one_subject("VP001", save_output=True, plot_figures=True)
#
#    print(f"number of epochs: {len(epochs)}")
#    print(f"label distribution: {np.bincount(labels)}")
#    print(f"data structure: {epochs.get_data().shape}")
    

In [22]:
# Batch processing the samples

def process_multiple_subjects(subject_range=range(1, 27), save_output=True, base_path=BASE_PATH):

    print(f"range: VP{min(subject_range):03d} - VP{max(subject_range):03d}")
    print(f"total: {len(subject_range)} subjects")
    
    results = {}
    
    for i in subject_range:
        subject_id = f"VP{i:03d}"
        print(f" processing {subject_id} ({i - min(subject_range) + 1}/{len(subject_range)})")

        try:
            epochs, labels, raw_clean = process_one_subject(
                subject_id, 
                save_output=save_output, 
                plot_figures=False  
            )
            
            results[subject_id] = {
                'n_trials': len(epochs),
                'labels_dist': np.bincount(labels).tolist(),
                'success': True
            } 
            print(f"{subject_id}: {len(epochs)} trials")
            
        except Exception as e:
            print(f"Error in processing sample {subject_id}: {e}")
            results[subject_id] = {'success': False, 'error': str(e)}

    # Printing the summary
    print("A summary of processing:")
    successful = sum(1 for r in results.values() if r.get('success', False))
    failed = len(subject_range) - successful
    
    print(f"successful: {successful} from {len(subject_range)}")
    print(f"failed: {failed} from {len(subject_range)}")
    
    if successful > 0:
        print("\n   subject are processed successfully:")
        for sid, info in results.items():
            if info.get('success', False):
                print(f"      {sid}: {info['n_trials']} trials, "
                      f"distribution: {info['labels_dist']}")
    
    if failed > 0:
        print("\n subjects with errors:")
        for sid, info in results.items():
            if not info.get('success', False):
                print(f"      {sid}: {info.get('error', 'Unknown error')}")
    
    return results

## Testing with a few subjects for debugging
#results = process_multiple_subjects(range(1, 2), save_output=True)

# results = process_multiple_subjects(range(1, 27), save_output=True)

In [23]:
# QCing the preprocessed samples

def check_data_quality(epochs, labels, subject_id):

    data = epochs.get_data()

    print(f"QC results of {subject_id}")

    print(f"Number of trials: {len(data)}")

    unique, counts = np.unique(labels, return_counts=True)
    print(f"\n2 class distibutions:")
    for lbl, cnt in zip(unique, counts):
        workload = {0: "0-back", 1: "2-back", 2: "3-back"}.get(lbl, "Unknown")
        print(f"   {workload}: {cnt} trials ({cnt/len(labels)*100:.1f}%)")

    # outlier detection
    channel_means = np.mean(data, axis=(0, 2))
    channel_stds = np.std(data, axis=(0, 2))

    print(f"Mean of amplitude in channels: {np.mean(channel_means):.2e} ± {np.std(channel_means):.2e}")
    print(f"SD of amplitude: {np.mean(channel_stds):.2e} ± {np.std(channel_stds):.2e}")

    # SNR checking
    signal_power = np.mean(data**2)
    noise_estimate = np.var(np.diff(data, axis=2)) / 2  
    snr = 10 * np.log10(signal_power / (noise_estimate + 1e-12))

    print(f"signal power: {signal_power:.2e}")
    print(f"SNR estimation: {snr:.1f} dB")

    if snr > 10:
        print("signal quality is good")
    elif snr > 0:
        print("signal quality is acceptable")
    else:
        print("signal quality is weak")
        
    # Saving the summary
    report_path = f'{subject_id}_quality_report.txt'
    with open(report_path, 'w') as f:
        f.write(f"Quality Report - {subject_id}\n")
        f.write("="*50 + "\n\n")
        f.write(f"Total trials: {len(data)}\n")
        f.write(f"Data shape: {data.shape}\n")
        f.write(f"Class distribution: {dict(zip(unique, counts))}\n")
        f.write(f"Mean amplitude: {np.mean(channel_means):.2e}\n")
        f.write(f"Std amplitude: {np.mean(channel_stds):.2e}\n")
        f.write(f"Signal power: {signal_power:.2e}\n")
        f.write(f"SNR: {snr:.1f} dB\n")

    print(f"\n processing summary is saving for {subject_id}_quality_report.txt")

    return {
        'snr': snr,
        'signal_power': signal_power,
        'channel_means': channel_means,
        'channel_stds': channel_stds
    }


# if 'subject_id' not in dir():
#    subject_id = "VP001"
    
# quality_metrics = check_data_quality(epochs_clean, labels_clean, subject_id)

# print(f"\n quality summary for {subject_id}")
# print(f"   SNR: {quality_metrics['snr']:.1f} dB")
# print(f"signal power: {quality_metrics['signal_power']:.2e}")


In [24]:
# Assessing fNIRS markers

subject_id = "VP001"
def inspect_fnirs_markers(subject_id, base_path=BASE_PATH):

    subject_nirs = f"{subject_id}-NIRS"
    mrk_path = os.path.join(base_path, subject_nirs, "mrk_nback.mat")

    print(f"fNIRS markers fro {subject_id}")

    if not os.path.exists(mrk_path):
        print(f"the folder is not found")
        return None

    mrk = loadmat(mrk_path)['mrk_nback'][0, 0]
    event_times_ms = mrk['time'].flatten()
    event_codes = mrk['event'][0, 0]['desc'].flatten().astype(int)

    print(f"total markers: {len(event_codes)}")
    print(f"unique markers: {sorted(np.unique(event_codes))}")

    unique, counts = np.unique(event_codes, return_counts=True)
    for code, count in zip(unique, counts):
        desc = {
            7: "0-back session start",
            8: "2-back session start",
            9: "3-back session start"
        }.get(code, "unknown")
        print(f"      marker {code}: {count} times ({desc})")

    return event_codes, event_times_ms

# fnirs_codes, fnirs_times = inspect_fnirs_markers(subject_id)


In [25]:
# Loading oxy and deoxy hemoglobin data from MAT files

FS_NIRS_ORIGINAL = 10.4  # Hz

def load_fnirs_raw_data(subject_id, base_path=BASE_PATH):

    subject_nirs = f"{subject_id}-NIRS"
    cnt_path = os.path.join(base_path, subject_nirs, "cnt_nback.mat")
    mnt_path = os.path.join(base_path, subject_nirs, "mnt_nback.mat")

    print(f"loading rwa data sample {subject_id}")
    print(f"cnt path: {cnt_path}")
    print(f"mnt path: {mnt_path}")

    if not os.path.exists(cnt_path):
        raise FileNotFoundError(f"path is not existed: {cnt_path}")

    # Loading
    cnt_data = loadmat(cnt_path)
    cnt = cnt_data['cnt_nback'][0, 0]
    
    mnt_data = loadmat(mnt_path)
    mnt = mnt_data['mnt_nback'][0, 0]
    
    # oxy and deoxy extraction
    if 'oxy' in cnt.dtype.names:
        oxy_raw = cnt['oxy'][0, 0]['x']
        deoxy_raw = cnt['deoxy'][0, 0]['x']
    else:
        oxy_raw = cnt['oxy']
        deoxy_raw = cnt['deoxy']
        
    # Sampling arte
    fn_nirs = FS_NIRS_ORIGINAL  # Hz (based on the description)
    #fs_nirs = float(cnt['fs'][0, 0])  

    print(f"main sampling rate: {fn_nirs} Hz")
    print(f"oxy daat structure: {oxy_raw.shape}")
    print(f"deoxy data shape: {deoxy_raw.shape}")
    if oxy_raw.ndim == 2:
        print(f"number of channels: {oxy_raw.shape[1]}")
        print(f"number of samples: {oxy_raw.shape[0]}")
    else: 
        print(f"number of channels is not accessible.")
        print(f"number of samples: {oxy_raw.shape[0]}")

    
    print(f"   range oxy: [{oxy_raw.min():.2e}, {oxy_raw.max():.2e}]")
    print(f"   range deoxy: [{deoxy_raw.min():.2e}, {deoxy_raw.max():.2e}]")
    
    print("Loading fNIRS data is now COMPELTED!")

    return oxy_raw, deoxy_raw, mnt, fn_nirs

# oxy_raw, deoxy_raw, mnt, fs_nirs = load_fnirs_raw_data(subject_id)


In [28]:
# Noise reduction for fNIRS using wavelet

def wavelet_denoising(data, alpha=0.1, wavelet='db5'):
    
    print(f"   Wavelet denoising (wavelet: {wavelet}, alpha: {alpha})...")
    
    cleaned_data = np.zeros_like(data)
    n_samples, n_channels = data.shape
    
    for ch in range(n_channels):
        signal = data[:, ch]
        level = min(int(np.floor(np.log2(n_samples))), 6)

        # wavelet decomposition
        coeffs = pywt.wavedec(signal, wavelet, level=level)
        new_coeffs = [coeffs[0]] 
   
        for i in range(1, len(coeffs)):
            d_coeffs = coeffs[i].copy()
            # noise estimation with MAD (Median Absolute Deviation)
            sigma = np.median(np.abs(d_coeffs - np.median(d_coeffs))) / 0.6745
            if sigma > 0:
                threshold = sigma * stats.norm.ppf(1 - alpha / 2)
                d_coeffs[np.abs(d_coeffs) > threshold] = 0
            new_coeffs.append(d_coeffs)

        # signal reconstruction
        cleaned_signal = pywt.waverec(new_coeffs, wavelet)
        cleaned_data[:, ch] = cleaned_signal[:n_samples]

    print(f"      output shape: {cleaned_data.shape}")
    return cleaned_data 


def batch_wavelet_denoising(data_list, alpha=0.1, wavelet='db5'):
    
    cleaned_list = []
    for i, data in enumerate(data_list):
        print(f"   Processing dataset {i+1}/{len(data_list)}...")
        cleaned = wavelet_denoising(data, alpha=alpha, wavelet=wavelet)
        cleaned_list.append(cleaned)
    
    return cleaned_list

In [29]:
# Motion artifact removal using MARA

def apply_mara(data, fs=10.4, window_size=41, threshold_factor=1.8):

    print(f"  applying MARA (window_size={window_size}, threshold_factor={threshold_factor})...")

    n_samples, n_channels = data.shape
    corrected_data = np.zeros_like(data)
    edge_len = int(0.5 * fs)
    
    for ch in range(n_channels):
        signal = data[:, ch].copy()

        # SD
        msd = pd.Series(signal).rolling(window=window_size, center=True, min_periods=1).std().to_numpy()

        # Theshold
        threshold = np.median(msd) + (threshold_factor * np.std(msd))
        ma_mask = (msd > threshold).astype(int)

        # artifact detection
        diff_mask = np.diff(ma_mask, prepend=0, append=0)
        starts = np.where(diff_mask == 1)[0]
        ends = np.where(diff_mask == -1)[0]
        indices = sorted(list(set([0, n_samples] + list(starts) + list(ends))))
        
        segments = [signal[indices[i]:indices[i+1]].copy() for i in range(len(indices) - 1)]

        if not segments:
            corrected_data[:, ch] = signal
            continue

        final_signal_parts = [segments[0]]
        for i in range(1, len(segments)):
            prev_seg = final_signal_parts[-1]
            curr_seg = segments[i]
            
            val_prev = np.median(prev_seg[-edge_len:]) if len(prev_seg) >= edge_len else prev_seg[-1]
            val_curr = np.median(curr_seg[:edge_len]) if len(curr_seg) >= edge_len else curr_seg[0]
            shift = val_prev - val_curr
            
            if ma_mask[indices[i]] == 1:
                # interpolation for artifact sections
                x = np.arange(len(curr_seg))
                s_val = max(np.var(curr_seg) * len(curr_seg), 1e-6)
                try:
                    spline = UnivariateSpline(x, curr_seg, s=s_val)
                    curr_seg = curr_seg - spline(x)
                    val_curr_new = np.median(curr_seg[:edge_len]) if len(curr_seg) >= edge_len else curr_seg[0]
                    curr_seg += (val_prev - val_curr_new)
                except:
                    curr_seg += shift
            else:
                if abs(shift) > 2 * np.std(msd):
                    curr_seg += shift
                    
            final_signal_parts.append(curr_seg)
        
        full_signal = np.concatenate(final_signal_parts)[:n_samples]
        
        t_idx = np.arange(n_samples)
        original_trend = np.polyval(np.polyfit(t_idx, data[:, ch], 1), t_idx)
        corrected_trend = np.polyval(np.polyfit(t_idx, full_signal, 1), t_idx)
        full_signal += (original_trend - corrected_trend)
        corrected_data[:, ch] = full_signal

        print(f"output strcuture: {corrected_data.shape}")

        return corrected_data

def batch_apply_mara(data_list, fs=10.4, window_size=41, threshold_factor=1.8):
    corrected_list = []
    for i, data in enumerate(data_list):
        print(f"   Processing dataset {i+1}/{len(data_list)}...")
        corrected = apply_mara(data, fs=fs, window_size=window_size, threshold_factor=threshold_factor)
        corrected_list.append(corrected)
    return corrected_list

def apply_mara_with_report(data, fs=10.4, window_size=41, threshold_factor=1.8, data_name="data"):
    print(f"\n   MARA for {data_name}:")
    corrected_data = apply_mara(data, fs=fs, window_size=window_size, threshold_factor=threshold_factor)
    reduction_ratio = 1 - (np.std(corrected_data) / (np.std(data) + 1e-12))
    print(f"      Artifact reduction ratio for {data_name}: {reduction_ratio:.2%}")   
    return corrected_data, reduction_ratio

def process_both_mara(oxy_data, deoxy_data, fs=10.4, window_size=41, threshold_factor=1.8):
    print("\n" + "=" * 60)
    print("Applying MARA (Motion Artifact Removal) to fNIRS data")
    print("=" * 60)
    
    oxy_corrected, ratio_oxy = apply_mara_with_report(oxy_data, fs, window_size, threshold_factor, "Oxy")
    deoxy_corrected, ratio_deoxy = apply_mara_with_report(deoxy_data, fs, window_size, threshold_factor, "Deoxy")

    print(f"   Oxy artifact reduction: {ratio_oxy:.2%}")
    print(f"   Deoxy artifact reduction: {ratio_deoxy:.2%}")

    return oxy_corrected, deoxy_corrected


In [30]:
# HbO and HbR concentrations

def calculate_hbo_hbr(oxy, deoxy, age=25, L=3.0):

    print(f"calculating HbO/HbR (age={age}, L={L} cm) ...")

    lambda1, lambda2 = 690, 830  # wave length (nm)

    # Differential Pathlength Factor
    def get_dpf(lmbda, age):
        alpha, beta, gamma = 223.3, 0.05624, 0.8493
        delta, eps, zeta = -5.723e-7, 0.001245, -0.9025
        return alpha + beta*(age**gamma) + delta*(lmbda**3) + eps*(lmbda**2) + zeta*lmbda
    
    dpf1 = get_dpf(lambda1, age)
    dpf2 = get_dpf(lambda2, age)

    # Molar extinction coef
    e1_hbo, e1_hbr = 0.3197, 1.3430   
    e2_hbo, e2_hbr = 1.0559, 0.6924  

    denom = L * (e1_hbr * e2_hbo - e2_hbr * e1_hbo)
    hbo = (e1_hbr * (deoxy/dpf2) - e2_hbr * (oxy/dpf1)) / denom
    hbr = (e1_hbo * (oxy/dpf1) - e2_hbo * (deoxy/dpf2)) / denom

    print(f"HbO shape: {hbo.shape}")
    print(f"HbR shape: {hbr.shape}")
    print(f"HbO range: [{hbo.min():.2e}, {hbo.max():.2e}]")
    print(f"HbR range: [{hbr.min():.2e}, {hbr.max():.2e}]")

    return hbo, hbr


def batch_calculate_hbo_hbr(oxy_list, deoxy_list, age=25, L=3.0):

    hbo_list = []
    hbr_list = []
    
    for i, (oxy, deoxy) in enumerate(zip(oxy_list, deoxy_list)):
        print(f"   Processing dataset {i+1}/{len(oxy_list)}...")
        hbo, hbr = calculate_hbo_hbr(oxy, deoxy, age=age, L=L)
        hbo_list.append(hbo)
        hbr_list.append(hbr)
    
    return hbo_list, hbr_list


def calculate_hbo_hbr_with_stats(oxy, deoxy, age=25, L=3.0, data_name="data"):

    print(f"\n   Processing {data_name}:")
    hbo, hbr = calculate_hbo_hbr(oxy, deoxy, age=age, L=L)
    
    stats = {
        'hbo_mean': np.mean(hbo),
        'hbo_std': np.std(hbo),
        'hbo_min': np.min(hbo),
        'hbo_max': np.max(hbo),
        'hbr_mean': np.mean(hbr),
        'hbr_std': np.std(hbr),
        'hbr_min': np.min(hbr),
        'hbr_max': np.max(hbr)
    }
    
    print(f"      HbO statistics: mean={stats['hbo_mean']:.2e}, std={stats['hbo_std']:.2e}")
    print(f"      HbR statistics: mean={stats['hbr_mean']:.2e}, std={stats['hbr_std']:.2e}")
    
    return hbo, hbr, stats


def process_both_hbo_hbr(oxy_data, deoxy_data, age=25, L=3.0):

    print("\n" + "=" * 60)
    print("Calculating HbO/HbR Concentrations (MBLL)")
    print("=" * 60)
    
    hbo_conc, hbr_conc = calculate_hbo_hbr(oxy_data, deoxy_data, age=age, L=L)
    
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.divide(hbo_conc, hbr_conc, where=(hbr_conc != 0))
        ratio_mean = np.mean(ratio[~np.isinf(ratio)])
    
    print(f"\n📊 HbO/HbR Ratio: {ratio_mean:.2f}")
    print("\n✅ HbO/HbR calculations completed!")
    
    return hbo_conc, hbr_conc

In [32]:
# Band-pass filter and downsampling rate

def apply_bandpass_filter(data, lowcut=0.01, highcut=0.1, fs=10.4, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    
    filtered_data = np.zeros_like(data)
    for ch in range(data.shape[1]):
        filtered_data[:, ch] = filtfilt(b, a, data[:, ch])
    
    return filtered_data

def resample_fnirs(data, original_fs=10.4, target_fs=10.0):
    n_samples = data.shape[0]
    new_n_samples = int(n_samples * (target_fs / original_fs))
    return resample(data, new_n_samples, axis=0)

def process_filter_and_downsample(hbo, hbr, fs=10.4, target_fs=10.0):
    print("   Applying bandpass filter (0.01-0.1 Hz)...")
    hbo_filtered = apply_bandpass_filter(hbo, fs=fs)
    hbr_filtered = apply_bandpass_filter(hbr, fs=fs)
    
    print("   Downsampling to 10 Hz...")
    hbo_out = resample_fnirs(hbo_filtered, original_fs=fs, target_fs=target_fs)
    hbr_out = resample_fnirs(hbr_filtered, original_fs=fs, target_fs=target_fs)
    
    print(f"      Output shapes: HbO={hbo_out.shape}, HbR={hbr_out.shape}")
    return hbo_out, hbr_out


def batch_filter_downsample(data_list, fs=10.4, target_fs=10.0):
    processed_list = []
    for i, data in enumerate(data_list):
        print(f"   Processing dataset {i+1}/{len(data_list)}...")
        filtered = apply_bandpass_filter(data, fs=fs)
        downsampled = resample_fnirs(filtered, original_fs=fs, target_fs=target_fs)
        processed_list.append(downsampled)
    return processed_list

In [39]:
# fNIRS: Spatial mapping to 16x16 grid

def spatial_mapping_16x16(continuous_data, mnt_struct):
    n_samples, n_channels = continuous_data.shape
    
    # Channels info
    x_coords = mnt_struct['x'].flatten()
    y_coords = mnt_struct['y'].flatten()
    
    print(f"   Coordinates x: [{x_coords.min():.2f}, {x_coords.max():.2f}]")
    print(f"   Coordinates y: [{y_coords.min():.2f}, {y_coords.max():.2f}]")
    
    # 16*16 grid
    grid_x, grid_y = np.mgrid[min(x_coords):max(x_coords):16j, 
                               min(y_coords):max(y_coords):16j]
    points = np.column_stack((x_coords, y_coords))
    
    print(f"   output grid dimensions: {grid_x.shape}")
    print(f"   number of input channels: {n_channels}")
    
    mapped_data = np.zeros((n_samples, 16, 16), dtype=np.float32)
    
    for t in range(n_samples):
        if t % 5000 == 0:
            print(f"      in processing {t}/{n_samples}...")
        values = continuous_data[t, :]
        grid_z = griddata(points, values, (grid_x, grid_y), method='cubic', fill_value=0)
        mapped_data[t] = grid_z
    
    print(f"   processing of {n_samples} sampels is completed!")
    return mapped_data

def process_both_spatial_mapping(hbo_data, hbr_data, mnt_struct, verbose=True):
    print("Spatial Mapping to 16x16 Grid")
    
    hbo_mapped = spatial_mapping_16x16(hbo_data, mnt_struct, verbose=verbose)
    hbr_mapped = spatial_mapping_16x16(hbr_data, mnt_struct, verbose=verbose)
    
    print(f"Spatial mapping completed!")
    print(f"   HbO shape: {hbo_mapped.shape}, Memory: {hbo_mapped.nbytes / 1024 / 1024:.1f} MB")
    print(f"   HbR shape: {hbr_mapped.shape}, Memory: {hbr_mapped.nbytes / 1024 / 1024:.1f} MB")
    
    return hbo_mapped, hbr_mapped


def batch_spatial_mapping(data_list, mnt_struct, verbose=False):
    mapped_list = []
    for i, data in enumerate(data_list):
        print(f"   Processing dataset {i+1}/{len(data_list)}...")
        mapped = spatial_mapping_16x16(data, mnt_struct, verbose=verbose)
        mapped_list.append(mapped)
    return mapped_list

In [40]:
# Saving processed data and final synchronization

def save_fnirs_processed_data(hbo_data, hbr_data, matched_sessions, subject_id, output_dir="PREPROCESSED_FNIRS"):

    os.makedirs(output_dir, exist_ok=True)

    print(f"saving fNIRS data for {subject_id}")

    np.save(f'{output_dir}/{subject_id}_hbo_16x16.npy', hbo_data)
    np.save(f'{output_dir}/{subject_id}_hbr_16x16.npy', hbr_data)

    print(f"   HbO saving: {output_dir}/{subject_id}_hbo_16x16.npy")
    print(f"   strcutue: {hbo_data.shape}")
    print(f"   HbR saving: {output_dir}/{subject_id}_hbr_16x16.npy")
    print(f"   strcuture: {hbr_data.shape}")

    #Offset calculating between EEg adn fNIRS
    offsets = []
    for sess in matched_sessions:
        offset = float(sess['eeg_time'] - sess['nirs_time'])
        offsets.append(offset)
    
    sessions_json = []
    for sess in matched_sessions:
        sessions_json.append({
            'label': int(sess['label']),
            'eeg_time': float(sess['eeg_time']),
            'nirs_time': float(sess['nirs_time']),
            'eeg_sample': int(sess['eeg_sample'])
        })
        
    # means
    if len(offsets) >= 27:
        avg_group1 = float(np.mean(offsets[0:9]))
        avg_group2 = float(np.mean(offsets[9:18]))
        avg_group3 = float(np.mean(offsets[18:27]))
    else:
        avg_group1 = float(np.mean(offsets))
        avg_group2 = float(np.mean(offsets))
        avg_group3 = float(np.mean(offsets))
    
    avg_offset_by_group = {
        'group1_16s': avg_group1,
        'group2_62s': avg_group2,
        'group3_110s': avg_group3
    }
    
    # Saving sessions and offsets info
    sessions_info = {
        'subject_id': str(subject_id),
        'n_sessions': int(len(matched_sessions)),
        'fs_nirs': float(10.0),
        'fs_eeg': float(200.0),
        'grid_size': int(16),
        'sessions': sessions_json,
        'offsets': {
            'individual': [float(x) for x in offsets],
            'average_by_group': {
                'group1_16s': avg_group1,
                'group2_62s': avg_group2,
                'group3_110s': avg_group3
            },
            'overall_mean': float(np.mean(offsets)),
            'overall_std': float(np.std(offsets))
        }
    }
    
    with open(f'{output_dir}/{subject_id}_sessions_info.json', 'w') as f:
        json.dump(sessions_info, f, indent=2)
    
    print(f" session info saving: {output_dir}/{subject_id}_sessions_info.json")
    
    # ofssets summary
    print(f"mean: group 1 (session1-9) = {avg_group1:.2f} sec")
    print(f"mean: group 2 (session10-18) = {avg_group2:.2f} sec")
    print(f"mean: group 3 (session19-27) = {avg_group3:.2f} sec")
    print(f"total mean: {np.mean(offsets):.2f} ± {np.std(offsets):.2f} sec")

    return sessions_info

def batch_save_fnirs_data(data_list, sessions_list, subject_ids, output_dir="PREPROCESSED_FNIRS"):
    all_info = {}
    for (hbo, hbr), sessions, sid in zip(data_list, sessions_list, subject_ids):
        info = save_fnirs_processed_data(hbo, hbr, sessions, sid, output_dir)
        all_info[sid] = info
    return all_info

In [43]:
FS_EEG = 200.0
EEG_SESSION_MARKERS = {112: 0, 128: 1, 144: 2}
FNIRS_SESSION_MARKERS = {7: 0, 8: 1, 9: 2}

def load_eeg_markers_for_sync(subject_id, base_path=BASE_PATH):
    subject_eeg = f"{subject_id}-EEG"
    mrk_path = os.path.join(base_path, subject_eeg, "mrk_nback.mat")
    
    print(f"EEG markers loading from {mrk_path}")
    
    if not os.path.exists(mrk_path):
        raise FileNotFoundError(f"no such file found: {mrk_path}")
        
    mrk = loadmat(mrk_path)['mrk_nback'][0, 0]
    event_times_ms = mrk['time'].flatten()
    event_codes = mrk['event'][0, 0]['desc'].flatten().astype(int)
    
    event_times_sec = event_times_ms / 1000.0
    eeg_samples = (event_times_sec * FS_EEG).astype(int)
    eeg_events = np.vstack([eeg_samples, np.zeros_like(eeg_samples), event_codes]).T
    
    return eeg_events, event_times_sec, event_codes


def match_sessions(eeg_events, fnirs_events, verbose=True):
    # پیدا کردن EEG sessions
    eeg_sessions = []
    for event in eeg_events:
        if event[2] in EEG_SESSION_MARKERS:
            eeg_sessions.append({
                'label': EEG_SESSION_MARKERS[event[2]],
                'sample': event[0],
                'time': event[0] / FS_EEG 
            })
    eeg_sessions.sort(key=lambda x: x['sample'])

    fnirs_sessions = []
    for time_ms, code in fnirs_events:
        if code in FNIRS_SESSION_MARKERS:
            fnirs_sessions.append({
                'label': FNIRS_SESSION_MARKERS[code],
                'time': time_ms / 1000.0  
            })
    fnirs_sessions.sort(key=lambda x: x['time'])

    print(f"EEG sessions: {len(eeg_sessions)}")
    print(f"fNIRS sessions: {len(fnirs_sessions)}")

    matched_sessions = []
    for i in range(min(len(eeg_sessions), len(fnirs_sessions))):
        if eeg_sessions[i]['label'] == fnirs_sessions[i]['label']:
            matched_sessions.append({
                'label': eeg_sessions[i]['label'],
                'eeg_time': eeg_sessions[i]['time'],
                'nirs_time': fnirs_sessions[i]['time'],
                'eeg_sample': eeg_sessions[i]['sample']
            })
    
    print(f"Number of matched sessions: {len(matched_sessions)}")
    
    labels = [s['label'] for s in matched_sessions]
    label_counts = Counter(labels)
    print(f"Label distribution: 0-back: {label_counts.get(0, 0)}, 2-back: {label_counts.get(1, 0)}, 3-back: {label_counts.get(2, 0)}")
    
    return matched_sessions, eeg_sessions, fnirs_sessions


print("Synchronization functions loaded successfully")

Synchronization functions loaded successfully


In [46]:
# Processing Loop

def run_processing_pipeline():
    
    subjects = get_subject_list()
    results = {}

    print(f"Processing {len(subjects)} subjects")

    for idx, subject_id in enumerate(subjects):
        subject_num = int(subject_id[2:])  # VP001 -> 1
        subject_path = os.path.join(BASE_PATH, f"{subject_num:03d}")

        eeg_folder = os.path.join(subject_path, f"{subject_id}-EEG")
        fnirs_folder = os.path.join(subject_path, f"{subject_id}-NIRS")

        if not os.path.exists(eeg_folder):
            print(f"EEG folder is not found: {eeg_folder}")
            results[subject_id] = {'success': False, 'error': 'EEG folder not found'}
            continue
            
        if not os.path.exists(fnirs_folder):
            print(f"fNIRS folder not found: {fnirs_folder}")
            results[subject_id] = {'success': False, 'error': 'fNIRS folder not found'}
            continue
        
        try:
            print(f"Processing {subject_id} ({idx+1}/{len(subjects)})")
            
            # EEG Processing
            print("EEG Processing...")
            raw, events, sfreq = load_data_with_markers(subject_id, base_path=subject_path)
            raw_clean, ica = preprocess_eeg_pipeline(raw, apply_ica=APPLY_ICA, verbose=VERBOSE)
            epochs, labels = extract_epochs_corrected(raw_clean, events, sfreq, 
                                                       tmin=EPOCH_TMIN, tmax=EPOCH_TMAX)

            if SAVE_OUTPUT:
                save_preprocessed_data(epochs, labels, subject_id, 
                                       output_dir=f"PREPROCESSED/{subject_id}")

            # fNIRS Processing
            print("fNIRS Processing...")
            oxy_raw, deoxy_raw, mnt, fs_nirs = load_fnirs_raw_data(subject_id, base_path=subject_path)

            fnirs_mrk_path = os.path.join(subject_path, f"{subject_id}-NIRS", "mrk_nback.mat")
            fnirs_mrk = loadmat(fnirs_mrk_path)['mrk_nback'][0, 0]
            fnirs_times = fnirs_mrk['time'].flatten()
            fnirs_codes = fnirs_mrk['event'][0, 0]['desc'].flatten().astype(int)

            # Wavelet denoising 
            oxy_wv = wavelet_denoising(oxy_raw)
            deoxy_wv = wavelet_denoising(deoxy_raw)
            
            # MARA 
            oxy_mara = apply_mara(oxy_wv)
            deoxy_mara = apply_mara(deoxy_wv)

            # HbO/HbR calculation
            hbo_conc, hbr_conc = calculate_hbo_hbr(oxy_mara, deoxy_mara)

            # Filtering and downsampling
            hbo_filtered = apply_bandpass_filter(hbo_conc)
            hbr_filtered = apply_bandpass_filter(hbr_conc)
            hbo_10hz = resample_fnirs(hbo_filtered)
            hbr_10hz = resample_fnirs(hbr_filtered)

            # Spatial mapping
            hbo_16x16 = spatial_mapping_16x16(hbo_10hz, mnt)
            hbr_16x16 = spatial_mapping_16x16(hbr_10hz, mnt)

            # Synchronization 
            fnirs_mrk_path = os.path.join(subject_path, f"{subject_id}-NIRS", "mrk_nback.mat")
            fnirs_mrk = loadmat(fnirs_mrk_path)['mrk_nback'][0, 0]
            fnirs_times = fnirs_mrk['time'].flatten()
            fnirs_codes = fnirs_mrk['event'][0, 0]['desc'].flatten().astype(int)
            fnirs_events = np.column_stack((fnirs_times, fnirs_codes))
            
            eeg_events, _, _ = load_eeg_markers_for_sync(subject_id, base_path=subject_path)
            
            matched_sessions, _, _ = match_sessions(eeg_events, fnirs_events, verbose=False)

            # Saving fNIRS results
            if SAVE_OUTPUT:
                save_fnirs_processed_data(hbo_16x16, hbr_16x16, matched_sessions, subject_id,
                                          output_dir=f"PREPROCESSED_FNIRS/{subject_id}")
            
            results[subject_id] = {
                'success': True,
                'eeg_trials': len(epochs),
                'fnirs_samples': hbo_16x16.shape[0],
                'sessions': len(matched_sessions)
            }
            
            print(f"\n{subject_id} completed successfully!")

        except Exception as e:
            print(f"\n Error in subject {subject_id}: {e}")
            import traceback
            traceback.print_exc()
            results[subject_id] = {'success': False, 'error': str(e)}

    # Summary  
    successful = sum(1 for r in results.values() if r.get('success', False))
    print(f"Successful: {successful} from {len(results)} subjects")
    print(f"Failed: {len(results) - successful} from {len(results)}")

    if successful > 0:
        print("\n Successfully processed subjects:")
        for sid, info in results.items():
            if info.get('success', False):
                print(f"   {sid}: EEG trials={info['eeg_trials']}, fNIRS samples={info['fnirs_samples']}, sessions={info['sessions']}")
    
    return results

In [49]:
# Run this script with this cell
if __name__ == "__main__":
    results = run_processing_pipeline()

Processing 2 subjects
Processing VP001 (1/2)
EEG Processing...
Loading Data VP001
Path: D:\Research\EEG-fNIRS\project\project\Dataset\001
EEG Sampling rate: 200.0 Hz
Number of channels: 30
Number of samples 394,930
Number of Events: 567
Existing Events: [16, 48, 64, 80, 96, 112, 128, 144]
Location for this channel is unknown; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
Loading is now COMPLETED!
Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 661 samples (3.305 s)



C:\Users\user\AppData\Local\Temp\ipykernel_15368\1157019149.py:62: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw.set_montage(mne.channels.make_dig_montage(ch_pos=ch_pos_dict),


Filtering raw data in 1 contiguous segment


[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done  29 out of  29 | elapsed:    0.3s finished


Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 49.38
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 49.12 Hz)
- Upper passband edge: 50.62 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 50.88 Hz)
- Filter length: 1321 samples (6.605 s)



...

Identified partions for removal:
         - Component ICA000: eye blink
         - Component ICA004: eye blink
         - Component ICA006: muscle artifact
         - Component ICA007: eye blink
         - Component ICA008: muscle artifact
         - Component ICA010: muscle artifact
no artifact is found
number of removed sections: 6
Applying ICA to Raw instance
    Transforming to ICA space (11 components)
    Zeroing out 6 ICA components
    Projecting back using 28 PCA components
Filtering raw data in 1 contiguous segment
Setting up low-pass filter at 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 59 samples (0.295 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    1.1s


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


[Parallel(n_jobs=1)]: Done  29 out of  29 | elapsed:    2.3s finished


Prepprocessing is now COMPLETED!
Epoch Extraction
Timing range: 0.0 - 3.0 sec after trial onset
epoch length: 3.0 sec
Number of samples per epoch: 600
trial events [16, 48, 64, 80, 96]
Number of extraction epochs: 540
Data shape: (540, 31, 601)
Number of sessions found: 27
session onset time (sec): [30.0, 99.445, 168.995, 238.495, 308.105]...

 number of trials without label: 0 from 540
      0-back: 180 trial (33.3%)
      2-back: 180 trial (33.3%)
      3-back: 180 trial (33.3%)
Saving preprocessed sample {subject_id}
data is saved: PREPROCESSED/VP002/VP002_epochs_data.npy
data structure: (540, 31, 601)
labels are saved: PREPROCESSED/VP002/VP002_labels.npy
distibution: [180 180 180]
Overwriting existing file.
Overwriting existing file.
data from VP002 is now SAVED!
data strcuture: (540, 31, 601)
label distibution: [180 180 180]
directory: PREPROCESSED/VP002/
fNIRS Processing...
loading rwa data sample VP002
cnt path: D:\Research\EEG-fNIRS\project\project\Dataset\002\VP002-NIRS\cnt_nb

In [1]:
import os
print("مسیر فعلی Jupyter:", os.getcwd())
print("فایل‌های موجود در مسیر فعلی:")
for f in os.listdir('.'):
    print(f"   - {f}")

مسیر فعلی Jupyter: C:\Users\user
فایل‌های موجود در مسیر فعلی:
   - .anaconda
   - .bash_history
   - .cache
   - .conda
   - .condarc
   - .continuum
   - .fop
   - .gitconfig
   - .IBM
   - .ipynb_checkpoints
   - .ipython
   - .jupyter
   - .keras
   - .lesshst
   - .matplotlib
   - .opera
   - .python_history
   - .spss
   - .spyder-py3
   - .texlive2022
   - .viminfo
   - .vscode
   - 3- Q8.ipynb
   - 3D Objects
   - 4- Q9.ipynb
   - anaconda3
   - ansel
   - AppData
   - Application Data
   - battery-report.html
   - CamScanner 05-26-2026 17.12.jpg
   - Computer Problems - Q8.ipynb
   - Computer Problems - Q9.ipynb
   - Contacts
   - Cookies
   - data_set_ALL_AML_independent.txt
   - data_set_ALL_AML_train.txt
   - Documents
   - Downloads
   - EEG-fNIRS_preprocessing.ipynb
   - EEG-fNIRS_preprocessing_oneSampleTest.ipynb
   - Favorites
   - filimo-dm.db
   - IntelGraphicsProfiles
   - Jedi
   - level1.ipynb
   - Links
   - Local Settings
   - model.ipynb
   - Music
   - My Docume